In [3]:
from dotenv import load_dotenv
from openai import OpenAI
import os
from pathlib import Path
import sys

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

# 1. Get the absolute path of the directory containing this script
parent_dir = Path(os.getcwd()).resolve().parent

# 3. Add the parent directory to Python's search path
sys.path.append(str(parent_dir))

##############################################################################
# 4. Import the function normally
from ingest import load_faq_data

# Use the function
documents = load_faq_data()
# Generate embedding for the documents
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

# batch processing to avoid memory issues
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

from rag_helper import RAGVector

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

# ANN search as an alternative to brute-force search:

NN (exact):    compare query against ALL documents -> top 5

ANN (approx):  narrow down to a region -> compare within region -> top 5

# sqlitesearch

Persistent sibling of minsearch. It stores vectors in SQLite, a real on-disk databse, and uses ANN strategies for retrieval.

In [4]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf", # sqlitesearch supports 3 ANN models: lsh, ivf, and hnsw
    db_path="faq_vectors2.db"
)

vs_index.fit(vectors, documents)

In [5]:
# Search:

query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [6]:
vs_index.close()

As the Vectors database is now available after the ingestion. We don't have to re-embed and index everytime the app starts. We only need to encode the query and then retrieve from this database.

In [7]:
# Set up the model
# Open the index
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

from rag_helper import RAGVector
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

vector_assistant.rag("the program has already begun, can I still sign up?")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

"Unfortunately, if the program has already begun, it's unlikely that you can still sign up for the course unless it's a very long course that spans multiple months or an online program that allows late registration. \n\nHowever, you can still reach out to the course administrators or program coordinators to inquire about the possibility of late registration or if there's an opportunity to join the course in a later cohort. They may provide more information on the course's flexible enrollment options, if any.\n\nThat being said, even if you can't join the course for this session, you can still express interest and ask about future course offerings, like the one planned for Summer 2027, as mentioned in the provided context."

In [8]:
# Close the connection:
vs_index.close()